[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihdicaballero/mento/blob/main/docs/source/examples/beam_summary_EN_1992-1-1.ipynb)

In [1]:
# Google Colab does not ship mento, so install it when running there.
# Guarded so that building the documentation never shells out to pip.
import sys

if "google.colab" in sys.modules:
    %pip install mento

# Beam summary with EN 1992-1-1

**Create input dataframe**

This example builds the two tables, sections and forces, directly in the cell; it is easier to read them from the sheets `Sections` and `Forces` of an Excel file with `from_excel`, as the User Guide explains.

In [2]:
from mento import MPa
from mento import BeamSummary, Concrete_EN_1992_2004, SteelBar
import pandas as pd

conc = Concrete_EN_1992_2004(name="H25", f_c=25 * MPa)
steel = SteelBar(name="ADN 500", f_y=500 * MPa)

# One row per section: its geometry, its stirrups (legs: 2 per closed stirrup) and the bars of each face.
# Reinforcement columns left out are zero.
sections = pd.DataFrame(
    {
        "Label": ["", "V101", "V102", "V103", "V104"],
        "b": ["cm", 20, 20, 20, 20],
        "h": ["cm", 50, 50, 50, 50],
        "cc": ["mm", 25, 25, 25, 25],
        "legs": ["", 0, 2, 2, 2],
        "dbs": ["mm", 0, 6, 6, 6],
        "sl": ["cm", 0, 20, 20, 20],
        "n1_top": ["", 2, 2, 2, 2],
        "db1_top": ["mm", 8, 12, 8, 8],
        "n2_top": ["", 0, 1, 0, 0],
        "db2_top": ["mm", 0, 16, 0, 0],
        "n1_bot": ["", 2, 2, 3, 3],
        "db1_bot": ["mm", 12, 8, 12, 12],
        "n2_bot": ["", 1, 0, 1, 0],
        "db2_bot": ["mm", 10, 0, 10, 0],
        "n3_bot": ["", 2, 0, 2, 0],
        "db3_bot": ["mm", 12, 0, 16, 0],
    }
)
# One row per load combination of a section: Nx > 0 compression, My > 0 bottom in tension.
forces = pd.DataFrame(
    {
        "Label": ["", "V101", "V102", "V103", "V104"],
        "Comb.": ["", "ELU 1", "ELU 2", "ELU 3", "ELU 4"],
        "Nx": ["kN", 0, 0, 0, 0],
        "Vz": ["kN", 20, -50, 100, 100],
        "My": ["kNm", 0, -35, 40, 45],
    }
)

beam_summary = BeamSummary(conc, steel, sections, forces)
# The sections table, as beam_summary.to_excel() writes it
beam_summary.sections_table

,Label,b,h,cc,legs,dbs,sl,n1_top,db1_top,n2_top,...,n4_top,db4_top,n1_bot,db1_bot,n2_bot,db2_bot,n3_bot,db3_bot,n4_bot,db4_bot
0,,cm,cm,mm,,mm,cm,,mm,,...,,mm,,mm,,mm,,mm,,mm
1,V101,20,50,25,0,0,0,2,8,0,...,0,0,2,12,1,10,2,12,0,0
2,V102,20,50,25,2,6,20,2,12,1,...,0,0,2,8,0,0,0,0,0,0
3,V103,20,50,25,2,6,20,2,8,0,...,0,0,3,12,1,10,2,16,0,0
4,V104,20,50,25,2,6,20,2,8,0,...,0,0,3,12,0,0,0,0,0,0


**Check all beams for shear and flexure**

In [3]:
beam_summary.check()

,Beam,b×h,"As,top","As,bot",Av,"Comb.,top","MEd,top","DCRb,top","Comb.,bot","MEd,bot","DCRb,bot","Comb.,v",VEd,NEd,DCRv,Warnings,Ok?
0,,cm,,,,,kNm,,,kNm,,,kN,kN,,,
1,V101,20×50,2Ø8,2Ø12+1Ø10 ++ 2Ø12,-,-,NaN,0.0,-,NaN,0.0,ELU 1,20.0,0.0,0.455,stirrups_required,❌
2,V102,20×50,2Ø12+1Ø16,2Ø8,1sØ6/20,ELU 2,-35.0,0.439,-,NaN,0.0,ELU 2,50.0,0.0,0.391,-,✅
3,V103,20×50,2Ø8,3Ø12+1Ø10 ++ 2Ø16,1sØ6/20,-,NaN,0.0,ELU 3,40.0,0.29,ELU 3,100.0,0.0,0.814,-,✅
4,V104,20×50,2Ø8,3Ø12,1sØ6/20,-,NaN,0.0,ELU 4,45.0,0.698,ELU 4,100.0,0.0,0.781,-,✅


**Calculate capacity for all beams for shear and flexure**

In [4]:
beam_summary.check(capacity_check=True)

,Beam,b×h,"As,top","As,bot",Av,"As,top,real","As,bot,real","Av,real","MRd,top","MRd,bot",VRd
0,,cm,,,,cm²,cm²,cm²/m,kNm,kNm,kN
1,V101,20×50,2Ø8,2Ø12+1Ø10 ++ 2Ø12,-,1.0,5.3,0.0,19.9,93.4,43.93
2,V102,20×50,2Ø12+1Ø16,2Ø8,1sØ6/20,4.3,1.0,2.8,79.7,20.0,127.81
3,V103,20×50,2Ø8,3Ø12+1Ø10 ++ 2Ø16,1sØ6/20,1.0,8.2,2.8,20.0,138.1,122.81
4,V104,20×50,2Ø8,3Ø12,1sØ6/20,1.0,3.4,2.8,20.0,64.5,128.07


**Export results to excel**

In [5]:
beam_summary.check().to_excel("Beam summary results.xlsx", index=False)

**See detailed results for shear check**

In [6]:
beam_summary.shear_results()
# See the same results for capacity check, without DCR
# beam_summary.shear_results(capacity_check=True)

,Label,Comb.,"Av,min","Av,req",Av,NEd,"VEd,1","VEd,2","VRd,c","VRd,s",VRd,"VRd,max","VEd,1≤VRd,max","VEd,2≤VRd",DCR
0,,,cm²/m,cm²/m,cm²/m,kN,kN,kN,kN,kN,kN,kN,,,
1,V101,ELU 1,1.6,1.6,0.0,0.0,20.0,20.0,43.93,0.0,43.93,43.93,True,True,0.455
2,V102,ELU 2,1.6,1.6,2.83,0.0,50.0,50.0,0.0,127.81,127.81,258.1,True,True,0.391
3,V103,ELU 3,1.6,2.3,2.83,0.0,100.0,100.0,0.0,122.81,122.81,248.0,True,True,0.814
4,V104,ELU 4,1.6,2.21,2.83,0.0,100.0,100.0,0.0,128.07,128.07,258.63,True,True,0.781


**See detailed results for flexure check**

In [7]:
beam_summary.flexure_results()
# See the same results for capacity check, without DCR
# beam_summary.flexure_results(capacity_check=True)

,Label,Comb.,Position,"As,min","As,req top","As,req bot",As,MEd,MRd,MEd≤MRd,DCR
0,,,,cm²,cm²,cm²,cm²,kNm,kNm,,
1,V101,ELU 1,Bottom,1.19,0.0,1.19,5.31,0.0,93.41,True,0.0
2,V102,ELU 2,Top,1.23,1.8,0.0,4.27,-35.0,79.74,True,0.439
3,V103,ELU 3,Bottom,1.18,0.0,2.15,8.2,40.0,138.12,True,0.29
4,V104,ELU 4,Bottom,1.24,0.0,2.33,3.39,45.0,64.46,True,0.698


**More detailed results**

In [8]:
# See more complete and specific detailed results for shear check for a specific beam index of the summary
beam_summary.nodes[1].shear_results_detailed()

===== BEAM SHEAR DETAILED RESULTS =====
Materials                                               Variable     Value  Unit
-----------------------------------------------------  ----------  -------  ------
Section Label                                                         V102
Concrete strength                                         fck           25  MPa
Steel reinforcement yield strength                        fywk         500  MPa
Safety factor for concrete                                 γc          1.5
Safety factor for steel                                    γs         1.15
Coefficient for long term effects and loading effects     αcc         0.85 

Geometry                     Variable     Value  Unit
--------------------------  ----------  -------  ------
Section height                  h            50  cm
Section width                   b            20  cm
Clear cover                     cc          2.5  cm
Longitudinal tension rebar      As         4.27  cm² 

Design forces

In [9]:
# See more complete and specific detailed results for shear check for a specific beam index of the summary
beam_summary.nodes[2].flexure_results_detailed()

===== BEAM FLEXURE DETAILED RESULTS =====
Materials                            Variable     Value  Unit
----------------------------------  ----------  -------  ------
Section Label                                      V103
Concrete strength                      fck           25  MPa
Steel reinforcement yield strength     fyk          500  MPa 

Geometry                  Variable     Value  Unit
-----------------------  ----------  -------  ------
Section height               h            50  cm
Section width                b            20  cm
Clear cover                  cc          2.5  cm
Mechanical top cover       cm,top        3.5  cm
Mechanical bottom cover    cm,bot        5.6  cm 

Design forces       Variable     Value  Unit
-----------------  ----------  -------  ------
Top max moment      MEd,top          0  kNm
Bottom max moment   MEd,bot         40  kNm 

Check                     Unit     Value  Min.    Max.    Ok?
-----------------------  ------  -------  ------  ------ 

**Export results to Word**

In [10]:
beam_summary.results_detailed_doc(index=1)

✅ Results exported to Beam_Summary_EN 1992-2004.docx


**Export results to Excel**

In [11]:
beam_summary.check().to_excel("Beam_summary_results.xlsx", index=False)